## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from paths import PROCESSED_DIR, RESULT_DIR

## Load the data file

In [ ]:
train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
test = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
X = pd.read_parquet(PROCESSED_DIR / 'X.parquet')
y = pd.read_parquet(PROCESSED_DIR / 'y.parquet')['reordered']
groups = pd.read_parquet(PROCESSED_DIR / 'groups.parquet')['user_id']
X_train = pd.read_parquet(PROCESSED_DIR / 'X_train.parquet')
y_train = pd.read_parquet(PROCESSED_DIR / 'y_train.parquet')['reordered']
X_test = pd.read_parquet(PROCESSED_DIR / 'X_test.parquet')
y_test = pd.read_parquet(PROCESSED_DIR / 'y_test.parquet')['reordered']

## Imports for evaluation

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
experiment_settings = [
    {
        'experiment': 'Default',
        'n_estimators': 500,
        'max_features': 'sqrt',
        'max_depth': None,
        'min_samples_split': 10,
        'min_samples_leaf': 5
    },
    {
        'experiment': 'Shallow Trees',
        'n_estimators': 500,
        'max_features': 'sqrt',
        'max_depth': 10,
        'min_samples_split': 10,
        'min_samples_leaf': 5
    },
    {
        'experiment': 'High Min Split and Leaf',
        'n_estimators': 500,
        'max_features': 'sqrt',
        'max_depth': None,
        'min_samples_split': 20,
        'min_samples_leaf': 10
    },
    {
        'experiment': 'Low Estimators',
        'n_estimators': 100,
        'max_features': 'sqrt',
        'max_depth': None,
        'min_samples_split': 10,
        'min_samples_leaf': 5
    },
    {
        'experiment': 'High Estimators',
        'n_estimators': 800,
        'max_features': 'sqrt',
        'max_depth': None,
        'min_samples_split': 10,
        'min_samples_leaf': 5
    },
    {
        'experiment': 'log2 Features',
        'n_estimators': 500,
        'max_features': 'log2',
        'max_depth': None,
        'min_samples_split': 10,
        'min_samples_leaf': 5
    }
]

In [ ]:
experiment_results = []

for setting in experiment_settings:
    model_exp = RandomForestClassifier(
        n_estimators=setting['n_estimators'],
        max_features=setting['max_features'],
        max_depth=setting['max_depth'],
        min_samples_split=setting['min_samples_split'],
        min_samples_leaf=setting['min_samples_leaf'],
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    )
    
    model_exp.fit(X_train, y_train)
    
    train_pred = model_exp.predict(X_train)
    test_pred = model_exp.predict(X_test)
    test_prob = model_exp.predict_proba(X_test)[:, 1]
    
    experiment_results.append({
        'Experiment': setting['experiment'],
        'n_estimators': setting['n_estimators'],
        'max_features': setting['max_features'],
        'max_depth': setting['max_depth'],
        'min_samples_split': setting['min_samples_split'],
        'min_samples_leaf': setting['min_samples_leaf'],
        'Train Accuracy': round(accuracy_score(y_train, train_pred), 4),
        'test Accuracy': round(accuracy_score(y_test, test_pred), 4),
        'Precision': round(precision_score(y_test, test_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, test_pred, zero_division=0), 4),
        'F1': round(f1_score(y_test, test_pred, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_test, test_prob), 4),
        'PR-AUC': round(average_precision_score(y_test, test_prob), 4)
    })

experiment_df = pd.DataFrame(experiment_results)
experiment_df

In [ ]:
experiment_df.sort_values('F1', ascending=False)

In [ ]:
experiment_df.to_csv(RESULT_DIR / 'random_forest_experiment_results.csv', index=False)
experiment_df

In [ ]:
tree_results = []

tree_list = [10, 50, 100, 200, 300, 500, 800]

for n_tree in tree_list:
    model_tree = RandomForestClassifier(
        n_estimators=n_tree,
        max_features='sqrt',
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    )
    
    model_tree.fit(X_train, y_train)
    test_pred_tree = model_tree.predict(X_test)
    
    tree_results.append({
        'n_estimators': n_tree,
        'F1': f1_score(y_test, test_pred_tree, zero_division=0),
        'Accuracy': accuracy_score(y_test, test_pred_tree)
    })

tree_df = pd.DataFrame(tree_results)
tree_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(tree_df['n_estimators'], tree_df['F1'], marker='o')
plt.title('Test F1 vs Number of Trees')
plt.xlabel('Number of Trees')
plt.ylabel('F1 Score')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
import pandas as pd

In [ ]:
rf = RandomForestClassifier(
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

param_grid = {
    'n_estimators': [100, 300],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [5, 10]
}

group_kfold = GroupKFold(n_splits=3)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='f1',
    cv=group_kfold,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search.fit(X, y, groups=groups)

In [ ]:
print("Best params:", grid_search.best_params_)
print("Best CV F1:", round(grid_search.best_score_, 4))

In [ ]:
grid_results_df = pd.DataFrame(grid_search.cv_results_)
grid_results_df = grid_results_df.sort_values('rank_test_score')

grid_results_df[[
    'rank_test_score',
    'mean_test_score',
    'param_n_estimators',
    'param_max_features',
    'param_max_depth',
    'param_min_samples_split',
    'param_min_samples_leaf'
]].head(10)

In [ ]:
rf = RandomForestClassifier(
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

param_grid = {
    'n_estimators': [100, 300],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [5, 10]
}

group_kfold = GroupKFold(n_splits=3)

scoring = {
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
    'accuracy': 'accuracy'
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=scoring,
    refit='f1',
    cv=group_kfold,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X, y, groups=groups)

grid_results_df = pd.DataFrame(grid_search.cv_results_)
grid_results_df = grid_results_df.sort_values('rank_test_f1')

grid_results_df[[
    'rank_test_f1',
    'mean_test_f1',
    'mean_test_precision',
    'mean_test_recall',
    'mean_test_accuracy',
    'param_n_estimators',
    'param_max_features',
    'param_max_depth',
    'param_min_samples_split',
    'param_min_samples_leaf'
]].head(10)

In [ ]:
features_imp = [c for c in X.columns]
brf_importance_df = pd.DataFrame({
    'feature': features_imp,
    'importance': grid_search.best_estimator_.feature_importances_
}).sort_values('importance', ascending=False)

brf_importance_df

In [ ]:
plt.figure(figsize=(8, 4))
top_grid_res = brf_importance_df.head(15).sort_values('importance')
plt.barh(top_grid_res['feature'], top_grid_res['importance'])
plt.title('Top Feature Importances')
plt.xlabel('Importance')
plt.show()

In [ ]:
grid_results_df.to_parquet(RESULT_DIR / 'grid_search_result.parquet', engine='fastparquet')

In [ ]:
grid_results_df.head()